In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../cluster.jl" )
include( "../visual.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )

home, base = expfolder( "system" );

In [ ]:
# Hyper parameters.
N = 1024
r = 1.0

# Speed and density parameters.
ρ = 0.01
s = 0.251
L = √(N/ρ)

# Extract folder name and extract critical noise.
# η = 0.1
folder = createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) )
η = readdlm( folder*"../order-0.5.txt" )[1]

# Create parameter variable.
params = Dict{Char,Real}( 'N' => N, 'L' => L, 'r' => r, 'v' => s, 'η' => η )

In [ ]:
# Initial conditions.
x = L*rand( N, )
y = L*rand( N, )
θ = 2*π*rand( N, ) .- π

# Time till "steady state".
T = 100_000;

In [ ]:
# Initialize progress meter.
meter = Progress( T, "Simulating to steady state."; dt=1.0 )

# Data variables.
φlist = Vector{defFloat}( undef, T + 1 )
φlist[1] = order( N, θ )

# Run simulation.
for t ∈ 1:T
    # Compute adjacency,
    A = adjacency( N, L, r, x, y, θ )

    # Iterate dynamics.
    x, y, θ = vicsek( params, A, x, y, θ )

    # Compute the order.
    φlist[t+1] = order( N, θ )

    # Update progress meter.
    next!( meter )
end

In [ ]:
# Plot the order parameter over time.
plt = plot(; size=(400,200) )

plot!( plt, 0:T, φlist; color=:black, lw=2, label="" )

plot!( plt; xlims=(0,T) )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel=L"t", ylabel=L"\varphi" )

In [ ]:
# Initialize steady state variables for saving.
Tsave = 10_000
φsave = Vector{defFloat}( undef, Tsave + 1 )

# Initial conditions.
φsave[1] = φlist[end];

In [ ]:
# Initialize cluster-related data variables.
f̂ = Vector{defInt}();
f = Dict{defInt,Vector{defInt}}( i => Vector{defInt}() for i ∈ 1:N )
c = zeros( defInt, N )

# Cluster information for initial conditions.
ιp = clusterinfo( N, adjacency( N, L, r, x, y, θ ) )[2];

In [ ]:
# Run simulation from steady state.
for t ∈ 1:Tsave
    # Compute the current adjacency matrix.
    A = adjacency( N, L, r, x, y, θ )

    # Extract cluster information from adjacency.
    n, ιc, _ = clusterinfo( N, A )
    push!( f̂, n... )

    # Update cluster distribution count.
    for ni ∈ n;  c[ni] += 1;  end

    # Extact the unique changes between the steps and update trackers.
    i, δi = uniquechanges( ιp, ιc )
    ιp = ιc

    # Add cluster change information to frequency list.
    countfreqlist!( N, f, i, δi )

    # Upadate the position of each agent.
    x, y, θ = vicsek( params, A, x, y, θ )

    # Compute the order parameter.
    φsave[t+1] = order( N, θ )
end

In [ ]:
# Extract cluster information for the last time step.
n, ιc, _ = clusterinfo( N, adjacency( N, L, r, x, y, θ ) )
i, δi = uniquechanges( ιp, ιc );

# Update trackers.
for ni ∈ n;  c[ni] += 1;  end
countfreqlist!( N, f, i, δi )

# Compute probability map and cluster distribution.
Δc = c./sum( c )
ω = probabilitymap( N, f );

In [ ]:
# Extract frequency map.
bins = -1/2:N-1/2
F = hcat( [fit( Histogram, f[i] .+ (i - 1), bins ).weights for i ∈ 1:N]... )';

In [ ]:
# Plot the cluster mass distribution.
plt = plot(; size=(400,250) )

i = Δc .> 0
scatter!( plt, (1:N)[i], Δc[i], color=:gray, label="" )

xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( Δc[i] ) ) )
plot!( plt, xlims=10.0.^[0,xmax], ylims=(10^ymin,10^0) )
plot!( plt, xscale=:log10, yscale=:log10 )

plot!( plt, xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )

In [ ]:
i = 25

# Plot the cluster mass distribution.
plt = plot(; size=(400,250) )

j = 0 .< F[i,:]
scatter!( plt, (1:N)[j], F[i,j]./sum( F[i,:] ), color=:gray, label="" )

xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( F[i,j]./sum( F[i,:] ) ) ) )
plot!( plt, xlims=10.0.^[0,xmax], ylims=(10^ymin,10^0) )
plot!( plt, xscale=:log10, yscale=:log10 )

plot!( plt, xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )

In [ ]:
# Save results of the simulation.
if false
    # Data folder name.
    println( createfolder( folder ) )

    # Save order parameter lists.
    writedlm( folder*"order_0-$(T).txt", φlist )
    writedlm( folder*"order_$(T)-end.txt", φsave )

    # Save cluster information.
    writedlm( folder*"frequency-map.txt", F )
end